# Análise do 99 de Krenko, Mob Boss

Este notebook **não** explica as cartas com lore de Commander nem com EDHREC. Ele reabre a lista em `data/solved_deck.txt` e decompõe o score que o `DeckSolver` realmente usou no fill/cut.

Fórmula v1 (sem teto de orçamento neste demo):

$$\mathrm{score} = 2\cdot\mathrm{synergy} + \mathrm{role\_need} - 1.4\cdot\mathrm{redundancy}$$

- **synergy:** Jaccard de tokens entre a carta e `query + texto do commander`. Se a carta veio do Chroma, usa o máximo entre Jaccard e $1/(1+d)$.
- **role_need:** bônus se o papel (land / ramp / draw / interaction / threat) ainda está abaixo da cota; penalidade se já estourou.
- **redundancy:** maior Jaccard (×1.25 se o papel coincide) contra o resto do 99.

**Caveat:** o greedy pontua *no momento da inserção*. Recalcular no 99 fechado subestima cartas que entraram cedo para preencher cota (o bônus de threat já não existe depois de 12 criaturas). Se existir `data/solved_deck_log.json` (gerado pelo `demo_solver.py`), a ordem e o score de inserção aparecem na tabela.

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from catalog import get_oracle_card
from deck_state import DeckState
from roles import ROLE_QUOTAS, classify_roles, role_counts
from rules_validator import CommanderValidator
from solver import DeckSolver

QUERY = "goblin tokens damage"
COMMANDER = "Krenko, Mob Boss"
LIST_PATH = ROOT / "data" / "solved_deck.txt"
SEED_PATH = ROOT / "data" / "test_deck.txt"
POOL_PATH = ROOT / "data" / "test_pool.txt"
LOG_PATH = ROOT / "data" / "solved_deck_log.json"
ATTACH_CHROMA = True  # False = só Jaccard/papéis, sem carregar MiniLM

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 80)
print("ROOT", ROOT)
print("lista", LIST_PATH, "existe" if LIST_PATH.exists() else "AUSENTE")

In [ ]:
def parse_list(path: Path) -> dict[str, int]:
    pattern = re.compile(r"^(\d+)x?\s+(.+)$")
    cards: dict[str, int] = {}
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            match = pattern.match(line)
            if not match:
                continue
            name = match.group(2).strip()
            cards[name] = cards.get(name, 0) + int(match.group(1))
    return cards


def origin_of(name: str, commander: str, seed: dict[str, int], pool: dict[str, int], insert_order: dict[str, int]) -> str:
    key = name.lower()
    if commander and key == commander.lower():
        return "commander"
    if key in {n.lower() for n in seed}:
        return "seed (test_deck)"
    if key in {n.lower() for n in pool}:
        return "pool (test_pool)"
    if key in insert_order:
        return "fill greedy"
    return "fill greedy"


def why_text(row: pd.Series) -> str:
    bits = []
    origin = row["origin"]
    if origin == "commander":
        return "Commander do demo. Não ocupa slot dos 99."
    if origin.startswith("seed"):
        bits.append("Já estava em test_deck.txt; o solver manteve.")
    elif origin.startswith("pool"):
        bits.append("Veio de test_pool.txt e venceu o greedy na hora de preencher.")
    else:
        bits.append("Candidata do retrieval (Chroma + queries de papel) escolhida pelo fill greedy.")
    if row.get("insert_rank") and pd.notna(row["insert_rank"]):
        bits.append(f"Inserida na posição {int(row['insert_rank'])} (score na hora {row.get('insert_score')}).")
    tokens = row.get("shared_tokens") or []
    if tokens:
        bits.append("Tokens em comum com commander+query: " + ", ".join(tokens[:12]) + ".")
    elif row["jaccard"] == 0 and not row.get("chroma_distance"):
        bits.append("Sem overlap textual com a query; entrou por cota de papel, curva ou fallback (ex.: Mountain).")
    if pd.notna(row.get("chroma_distance")):
        q = row.get("chroma_query") or "?"
        bits.append(
            f"Vizinha no embedding (d={row['chroma_distance']:.3f}, sinergia {row['chroma_synergy']:.3f}) via «{q}»."
        )
    roles = row.get("roles") or []
    rs = row["role_score"]
    if rs >= 2:
        bits.append(f"Papel {', '.join(roles)} ainda abaixo da cota mínima neste recálculo.")
    elif rs < 0:
        bits.append(f"Papel {', '.join(roles)} saturado no 99 fechado (penalidade). No insert o bônus pode ter sido alto.")
    if row["redundancy"] >= 0.45 and row.get("redundancy_with"):
        bits.append(f"Redundante com {row['redundancy_with']} ({row['redundancy']:.2f}).")
    if row.get("curve_penalty"):
        bits.append("CMC desta faixa já tinha ≥18 não-terrenos no 99 fechado.")
    bits.append(
        f"Score pós-hoc {row['total']:.3f} = 2×{row['synergy']:.3f} + {row['role_score']:.3f} − 1.4×{row['redundancy']:.3f}."
    )
    return " ".join(bits)

In [ ]:
raw = parse_list(LIST_PATH)
seed = parse_list(SEED_PATH) if SEED_PATH.exists() else {}
pool = parse_list(POOL_PATH) if POOL_PATH.exists() else {}
log = json.loads(LOG_PATH.read_text(encoding="utf-8")) if LOG_PATH.exists() else {}
if log.get("query"):
    QUERY = log["query"]

insert_order: dict[str, int] = {}
insert_score: dict[str, float] = {}
for i, item in enumerate(log.get("fill", {}).get("added") or [], start=1):
    name = item["name"]
    key = name.lower()
    insert_order.setdefault(key, i)
    insert_score.setdefault(key, item.get("score"))

commander = log.get("commander") or COMMANDER
main = {n: q for n, q in raw.items() if n.lower() != commander.lower()}
cmd_info = get_oracle_card(commander)

deck = DeckState(
    commander=commander,
    cards=main,
    intent="improve",
    require_complete=True,
    identity=list((cmd_info or {}).get("color_identity") or ["R"]),
)
if cmd_info:
    deck.set_commander(commander)

solver = DeckSolver()
if ATTACH_CHROMA:
    solver._retrieve(deck, QUERY)
solver._rebuild_context(deck, QUERY)

validation = CommanderValidator().validate_deck_state(deck)
print("query:", QUERY)
print("slots", deck.slot_count(), "valid", validation.get("valid"), "complete", validation.get("complete"))
print("warnings", validation.get("warnings"))
print("seed", seed)
print("log fill adds", len(log.get("fill", {}).get("added") or []), "swaps", len(log.get("cut", {}).get("swapped") or []))

In [ ]:
rows = []
cmd_row = solver.score_breakdown(deck, commander, QUERY)
cmd_row["quantity"] = 1
cmd_row["origin"] = "commander"
cmd_row["insert_rank"] = None
cmd_row["insert_score"] = None
rows.append(cmd_row)

for name, qty in sorted(deck.card_list().items(), key=lambda kv: kv[0].lower()):
    br = solver.score_breakdown(deck, name, QUERY)
    br["quantity"] = qty
    br["origin"] = origin_of(name, commander, seed, pool, insert_order)
    br["insert_rank"] = insert_order.get(name.lower())
    br["insert_score"] = insert_score.get(name.lower())
    rows.append(br)

analysis = pd.DataFrame(rows)
analysis["why"] = analysis.apply(why_text, axis=1)
analysis["roles"] = analysis["roles"].map(lambda xs: ", ".join(xs) if isinstance(xs, list) else xs)
analysis["shared_tokens"] = analysis["shared_tokens"].map(
    lambda xs: ", ".join(xs) if isinstance(xs, list) else xs
)

view = analysis[
    [
        "origin",
        "quantity",
        "name",
        "cmc",
        "roles",
        "total",
        "synergy",
        "jaccard",
        "chroma_distance",
        "chroma_query",
        "role_score",
        "redundancy",
        "redundancy_with",
        "insert_rank",
        "insert_score",
        "why",
    ]
].sort_values(["origin", "total"], ascending=[True, False])

view

## Por que cada carta (texto gerado do score)

A coluna `why` é derivada dos componentes acima. Cartas **seed** (Mountain, Sol Ring) não foram “escolhidas” pelo modelo — só não foram cortadas. Goblins com `chroma_query` próxima do oracle do Krenko ou de `creature tokens` são o núcleo geométrico da lista.

In [ ]:
why_view = analysis.loc[analysis["origin"] != "commander", ["quantity", "name", "origin", "total", "why"]]
why_view = why_view.sort_values("total", ascending=False)
pd.set_option("display.max_colwidth", 400)
why_view

## Cotas de papel vs o 99

`ROLE_QUOTAS`: land 34–38, ramp/draw 8–14, interaction 8–15, threat 12–40. Uma carta pode ter vários papéis (criatura que causa dano = threat + interaction).

In [ ]:
infos = []
for name, qty in deck.card_list().items():
    info = solver._info(name) or {"name": name, "type_line": "", "oracle_text": ""}
    infos.append({**info, "quantity": qty})
counts = role_counts(infos)
quota_rows = []
for role, (lo, hi) in ROLE_QUOTAS.items():
    n = counts.get(role, 0)
    if n < lo:
        status = "abaixo do mínimo"
    elif n > hi:
        status = "acima do máximo"
    else:
        status = "dentro da cota"
    quota_rows.append({"role": role, "count": n, "min": lo, "max": hi, "status": status})
quota_rows.append({"role": "other", "count": counts.get("other", 0), "min": None, "max": None, "status": "sem cota"})
quota_df = pd.DataFrame(quota_rows)
display(quota_df)

print("\nCurva (não-terrenos) e lands no validador:")
print("curve", validation.get("curve"))
print("land_count", validation.get("land_count"), "budget_used", validation.get("budget_used"))

In [ ]:
import matplotlib.pyplot as plt

body = analysis[analysis["origin"] != "commander"].copy()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

qplot = quota_df[quota_df["role"] != "other"]
axes[0, 0].bar(qplot["role"], qplot["count"], color="#c44e52", label="no 99")
axes[0, 0].plot(qplot["role"], qplot["min"], "ko", label="mínimo")
axes[0, 0].plot(qplot["role"], qplot["max"], "k^", label="máximo")
axes[0, 0].set_title("Papéis no 99 vs cotas do solver")
axes[0, 0].set_ylabel("cartas (com multi-papel)")
axes[0, 0].legend()

axes[0, 1].hist(body["total"], bins=12, color="#4c72b0", edgecolor="white")
axes[0, 1].set_title("Score pós-hoc no 99 fechado")
axes[0, 1].set_xlabel("score")
axes[0, 1].set_ylabel("cartas únicas")

curve = validation.get("curve") or {}
order = ["0", "1", "2", "3", "4", "5", "6", "7+"]
axes[1, 0].bar(order, [curve.get(k, 0) for k in order], color="#55a868")
axes[1, 0].set_title("Curva de CMC (não-terrenos)")
axes[1, 0].set_xlabel("CMC")
axes[1, 0].set_ylabel("cartas")

axes[1, 1].scatter(body["synergy"], body["redundancy"], c=body["total"], cmap="coolwarm", s=28)
axes[1, 1].set_title("Sinergia vs redundância")
axes[1, 1].set_xlabel("synergy")
axes[1, 1].set_ylabel("redundancy")

fig.suptitle("Krenko 99 — diagnóstico do solver (não é EDHREC)")
fig.tight_layout()
plt.show()

## Extremos: o que o greedy mais “quis” vs o que só ocupou slot

No 99 fechado, scores altos ainda carregam overlap com goblin/token/damage. Scores baixos são seed (Mountain), artefactos genéricos puxados por queries de ramp/draw, ou cartas cujo papel já estava cheio.

In [ ]:
ranked = body.sort_values("total", ascending=False)
print("Top 12 (melhor score pós-hoc)")
display(ranked.head(12)[["quantity", "name", "roles", "total", "synergy", "chroma_query", "insert_rank"]])
print("Bottom 12 (pior score pós-hoc — candidatas naturais a cut)")
display(ranked.tail(12)[["quantity", "name", "roles", "total", "synergy", "redundancy_with", "origin"]])

## Ficha de uma carta

Troque o nome para ver oracle, papéis e o texto de escolha.

In [ ]:
def show_card(name: str) -> pd.Series:
    hit = analysis[analysis["name"].str.lower() == name.lower()]
    if hit.empty:
        raise KeyError(f"{name!r} não está na lista/commander")
    row = hit.iloc[0]
    print(row["name"], "—", row["type_line"])
    print(row.get("mana_cost") or "", "CMC", row["cmc"], "USD", row.get("price_usd"))
    print("papéis:", row["roles"])
    print("oracle:")
    print(row["oracle_text"] or "(sem texto)")
    print()
    print(row["why"])
    return row

show_card("Goblin Warchief")